In [23]:
#%pip install git+https://github.com/holehouse-lab/shephard.git
import numpy as np


In [2]:
from shephard.apis import uniprot
from shephard import interfaces

In [3]:
filename = '/home/wenyuantong/Desktop/data/UP000006548_3702.fasta'
#fasta is arabidopsis thaliana

In [4]:

#arabi_proteome = uniprot._deal_with_invalid_sequences(filename)
#editied in vim from 'fail' to 'ignore'

In [5]:
arabi_proteome = uniprot.uniprot_fasta_to_proteome(filename)

In [6]:
# excise and assign gene names as attributes
for protein in arabi_proteome:
    name_string = protein.name
    
    # first try and get gene name from the GN= entry in the FASTA header
    try:
        gene_name = name_string.split('GN=')[1].split()[0]
    except IndexError:
        # if this fails get the UID-assigned identifier which is 
        # typicall gene-name (or close to) followed by species identifier
        # - e.g. P53_HUMAN in instead of TP53. We keep the _HUMAN so records
        # parsed in this way can be easily identified (although) the _HUMAN
        # could be excised 
        gene_name = name_string.split()[0].split('|')[2]
        
    protein.add_attribute('gene_name', gene_name)

In [8]:
uid_of_interest = 'P04637'

print(arabi_proteome.protein(uid_of_interest).attribute('gene_name'))

ProteomeException: unique_ID 'P04637' not found in proteome

In [17]:
# the line here writes the gene name annotations out to a SHEPHARD protein attributes file
interfaces.si_protein_attributes.write_protein_attributes(arabi_proteome,'/home/wenyuantong/Desktop/data/shprd_prot_atts_gene_names.tsv')

In [18]:
# Define first 5 residues of every protien as C and N terminal domains
for p in arabi_proteome:
    p.add_domain(1,5, 'N-Terminus')
    p.add_domain(p._len-4,p._len, 'C-Terminus')

In [20]:
# anylize C and N terminal domains
for d in arabi_proteome.domains:
    if d.domain_type in ['N-Terminus','C-Terminus']:
        d.add_attribute('f_G', d.sequence.count('G')/len(d.sequence))
        d.add_attribute('f_S', d.sequence.count('S')/len(d.sequence))

In [24]:
# Caluclate fraction of Gly & Ser in C and N terminal regions
N_terms = [(d.attribute('f_G'), d.attribute('f_S')) for d in arabi_proteome.domains if d.domain_type == 'N-Terminus']
C_terms = [(d.attribute('f_G'), d.attribute('f_S')) for d in arabi_proteome.domains if d.domain_type == 'C-Terminus']

print('Average N Termainal fractions: G:%.3f  S:%.3f' % tuple(map(np.mean, zip(*N_terms))))
print('Average C Termainal fractions: G:%.3f  S:%.3f' % tuple(map(np.mean, zip(*C_terms))))

Average N Termainal fractions: G:0.050  S:0.102
Average C Termainal fractions: G:0.051  S:0.104


In [26]:
# annotate list of proteins with C or N terminal poly-GS
poly_GS_termi_proteins = [d.protein for d in arabi_proteome.domains if d.attribute('f_G') + d.attribute('f_S') == 1]

print('Number of protiens with poly-GS C-Terminal or N-Terminal:', len(poly_GS_termi_proteins))
print('Proteins:', [p.unique_ID for p in poly_GS_termi_proteins])

Number of protiens with poly-GS C-Terminal or N-Terminal: 30
Proteins: ['Q9ZUH3', 'Q9LMR3', 'Q9SD98', 'Q9LSP9', 'A0A1I9LTU3', 'Q9FME2', 'Q9LVX0', 'Q9LHF0', 'F4I045', 'Q9MA41', 'Q84WW4', 'Q9SFB0', 'Q9SZZ8', 'Q1PFW1', 'Q9FIW9', 'Q9XIP5', 'Q8H1G4', 'Q9ZT42', 'Q9FJS3', 'O80548', 'O49447', 'Q9SEZ1', 'A0A1I9LPW1', 'Q8GYK2', 'Q9M1U3', 'Q8L7S7', 'Q9LTA3', 'Q6DR24', 'A0JQ78', 'Q9LPH9']


In [30]:
from shephard.interfaces import si_domains, si_sites

In [31]:
si_domains.add_domains_from_file(arabi_proteome, '/home/wenyuantong/Desktop/data/filtered_IDR_Arabi.csv')

In [33]:
for protein in arabi_proteome:
    for domain_idx in protein.domains:
        domain = protein.domain(domain_idx)
        if domain.start ==  1:
            print(f'Protein {protein} has an N-terminal domain: {domain}')

ProteinException: No domains named [|Domain: N-Terminus (1-5, len=5) in protein A0A1I9LPH3] in protein A0A1I9LPH3

Available domains are: ['N-Terminus_1_5', 'C-Terminus_46_50']

In [34]:
arabi_proteome

[Proteome]: Sequence dataset with 27448 protein records

In [40]:
arabi_proteome[0:20]

[| Protein: A0A1I9LPH3 - L=50, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: A0A1P8B415 - L=854, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: A4FVP2 - L=565, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: B3H6N5 - L=37, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4I2G0 - L=107, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4I316 - L=660, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4I8H8 - L=328, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4IB23 - L=225, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4IB81 - L=651, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4IG53 - L=125, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4IRU3 - L=1556, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4IYZ0 - L=358, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4J3U4 - L=165, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4J9Q6 - L=499, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4JN22 - L=439, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4JRR2 - L=1122, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4JTS9 - L=404, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4K4Y5 - L=778, #t=0, #d=2, #s=0, #a=1 |,
 | Protein: F4K6R7 -